In [1]:
import pandas as pd

df = pd.read_csv('data/flipkart_product.csv', encoding='latin1', on_bad_lines='skip')
df = df[['ProductName', 'Rate', 'Summary']].dropna(subset=['Summary'])
df = df.rename(columns={'Summary': 'review_text'})

print(df.shape)
df.head()

(189860, 3)


,ProductName,Rate,review_text
0,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,5,Great cooler.. excellent air flow and for this...
1,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,5,Best budget 2 fit cooler. Nice cooling
2,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,3,The quality is good but the power of air is de...
3,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,1,Very bad product it's a only a fan
4,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,3,Ok ok product


In [2]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)          # remove links
    text = re.sub(r'[^a-z\s]', '', text)          # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()      # collapse extra spaces
    return text

df['clean_review'] = df['review_text'].apply(clean_text)

# Also clean product name for display purposes (remove the encoding junk)
df['ProductName'] = df['ProductName'].str.replace(r'[^\x00-\x7F]+', '', regex=True).str.strip()

print(df[['review_text', 'clean_review']].head(5))

                                         review_text  \
0  Great cooler.. excellent air flow and for this...   
1             Best budget 2 fit cooler. Nice cooling   
2  The quality is good but the power of air is de...   
3                 Very bad product it's a only a fan   
4                                      Ok ok product   

                                        clean_review  
0  great cooler excellent air flow and for this p...  
1                best budget fit cooler nice cooling  
2  the quality is good but the power of air is de...  
3                  very bad product its a only a fan  
4                                      ok ok product  


In [3]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['clean_review'].apply(get_sentiment)

sentiment_pct = df['sentiment'].value_counts(normalize=True) * 100
print(sentiment_pct.round(2))

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...


sentiment
Positive    79.21
Negative    11.60
Neutral      9.19
Name: proportion, dtype: float64


In [4]:
complaint_keywords = {
    "Delivery delays": ["late", "delay", "delivery", "shipping", "arrived", "shipment"],
    "Packaging issues": ["package", "packaging", "damaged", "box", "broken", "torn"],
    "Customer support problems": ["support", "refund", "response", "service", "helpline", "replace", "return"]
}

def tag_complaints(text):
    tags = []
    for category, keywords in complaint_keywords.items():
        if any(kw in text for kw in keywords):
            tags.append(category)
    return tags

df['complaints'] = df['clean_review'].apply(tag_complaints)

# Count how many reviews mention each complaint category
complaint_counts = df['complaints'].explode().value_counts()
print(complaint_counts)

complaints
Delivery delays              5373
Customer support problems    4287
Packaging issues             2760
Name: count, dtype: int64


In [5]:
def generate_summary(sentiment_pct, complaint_counts):
    top_complaint = complaint_counts.idxmax()
    positive_pct = sentiment_pct.get('Positive', 0)
    negative_pct = sentiment_pct.get('Negative', 0)
    
    summary = (
        f"{positive_pct:.0f}% of customers had a positive experience overall, "
        f"but satisfaction is impacted mainly by {top_complaint.lower()}, "
        f"which was the most frequently mentioned issue "
        f"({complaint_counts.max():,} mentions)."
    )
    return summary

ai_summary = generate_summary(sentiment_pct, complaint_counts)
print(ai_summary)

79% of customers had a positive experience overall, but satisfaction is impacted mainly by delivery delays, which was the most frequently mentioned issue (5,373 mentions).


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

sample_reviews = df['review_text'].dropna().sample(50, random_state=42).tolist()
combined_text = " ".join(sample_reviews)[:3000]

inputs = tokenizer(combined_text, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=20,
    num_beams=4,
    early_stopping=True
)
transformer_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(transformer_summary)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

 The chimney looks great and works like a charm . It does have a constant whirring sound when turned on but I think all chimneys do have background noise . The installation was done within a day .


In [8]:
# Sample from reviews that mention the delivery delay complaint specifically
delivery_reviews = df[df['complaints'].apply(lambda x: "Delivery delays" in x)]['review_text']
sample_reviews = delivery_reviews.dropna().sample(min(50, len(delivery_reviews)), random_state=42).tolist()
combined_text = " ".join(sample_reviews)[:3000]

inputs = tokenizer(combined_text, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=20,
    num_beams=4,
    early_stopping=True
)
transformer_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(transformer_summary)

 Plates are very light weighted and good design but bowls are very small in size . Table and quater plate are awesome but mini bowl is very very mini... Other wise product is good u can buy it. The delivery from Flipkart was on time .


In [9]:
df.to_csv('data/processed_reviews.csv', index=False)
print("Saved:", df.shape)

Saved: (189860, 6)


In [10]:
import streamlit as st
import pandas as pd
import plotly.express as px
import ast

st.set_page_config(page_title="Customer Review Intelligence", layout="wide")

st.title("📊 Customer Review Intelligence Dashboard")
st.caption("Flipkart Product Reviews — Sentiment, Complaints & AI Summary")

# Load processed data
df = pd.read_csv('data/processed_reviews.csv')

# complaints column was saved as a string like "['Delivery delays']" — convert back to a list
df['complaints'] = df['complaints'].apply(ast.literal_eval)

# ---- Sentiment Section ----
col1, col2 = st.columns(2)

with col1:
    st.subheader("Sentiment Breakdown")
    sentiment_counts = df['sentiment'].value_counts()
    fig1 = px.pie(
        values=sentiment_counts.values,
        names=sentiment_counts.index,
        color=sentiment_counts.index,
        color_discrete_map={"Positive": "#2ecc71", "Negative": "#e74c3c", "Neutral": "#95a5a6"}
    )
    st.plotly_chart(fig1, use_container_width=True)

with col2:
    st.subheader("Main Complaints")
    complaint_flat = df['complaints'].explode().dropna().value_counts()
    fig2 = px.bar(
        x=complaint_flat.values,
        y=complaint_flat.index,
        orientation='h',
        labels={'x': 'Number of Mentions', 'y': ''}
    )
    st.plotly_chart(fig2, use_container_width=True)

# ---- AI Summary Section ----
st.subheader("🤖 AI Summary")

sentiment_pct = df['sentiment'].value_counts(normalize=True) * 100
top_complaint = complaint_flat.idxmax()
positive_pct = sentiment_pct.get('Positive', 0)

summary = (
    f"{positive_pct:.0f}% of customers had a positive experience overall, "
    f"but satisfaction is impacted mainly by {top_complaint.lower()}, "
    f"which was the most frequently mentioned issue "
    f"({complaint_flat.max():,} mentions)."
)
st.info(summary)

# ---- Raw Data Explorer ----
with st.expander("🔍 Explore raw reviews"):
    selected_sentiment = st.selectbox("Filter by sentiment", ["All"] + list(df['sentiment'].unique()))
    display_df = df if selected_sentiment == "All" else df[df['sentiment'] == selected_sentiment]
    st.dataframe(display_df[['ProductName', 'Rate', 'review_text', 'sentiment']].head(200))

2026-08-12 23:59:13.177 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 23:59:13.178 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 23:59:13.317 
  command:

    streamlit run c:\Users\shrey\OneDrive\Dokumenty\Customer_review_intelligence_system\venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-08-12 23:59:13.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 23:59:13.321 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 23:59:13.323 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 23:59:13.324 Thread 'MainThread': missing Scrip

In [13]:
import streamlit as st
import pandas as pd
import plotly.express as px
import ast

st.write("✅ App is running")

st.set_page_config(page_title="Customer Review Intelligence", layout="wide")

2026-08-13 00:10:55.755 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:10:55.757 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:10:55.759 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:10:55.760 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [14]:
import streamlit as st
import pandas as pd
import plotly.express as px
import ast

st.set_page_config(page_title="Customer Review Intelligence", layout="wide")

st.write("✅ App is running")

st.title("📊 Customer Review Intelligence Dashboard")
...

2026-08-13 00:11:04.545 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:11:04.547 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:11:04.547 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:11:04.549 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:11:04.551 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:11:04.552 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 00:11:04.554 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Ellipsis